# `quadbench`

This notebook reports three benchmark runs: two on a single-socket x86-64 machine
and one on an NVIDIA A100. Timings were collected by `bench_v2.py` (CPU) and
`bench_gpu.py` (GPU) and stored as per-repeat microsecond samples in `.npz`
archives. `clean_npz.py` converts each archive into a tidy Parquet table; the
analysis below reads those tables together with the metadata JSON written
alongside them.

| run | subject | shape | dtypes |
|---|---|---|---|
| `cpu_sweep` | cache hierarchy and precision, three problem sizes | n = 500 / 4 000 / 64 000 | 13, including `quad-sleef` and x87 `longdouble` |
| `cpu_dram` | streaming past every cache level, one large size | n = 2 000 000 | 4 |
| `gpu_sweep` | A100 HBM bandwidth and fp16/32/64 throughput, four sizes | n = 20 000 … 25 000 000 | 3 |

Every measured cell is defined by four factors:

- **`destination`** — `alloc` allocates a fresh result array, as an ordinary
  Python expression would; `out` writes into a preallocated buffer. The
  difference between the two isolates the cost of allocation.
- **`cache_state`** — `hot` reuses a single operand pair, which therefore remains
  cache-resident; `cold` cycles through distinct pairs so that every call streams
  its operands from memory.
- **`n`** — problem size. Arrays have shape `(n, 2, 2)`, so `n_elem = 4n`.
- **`implementation`** — the operand dtype. NumPy may promote the operands before
  computing; the metadata records the true result dtype, which is used for byte
  accounting throughout in preference to the operand dtype.

Timings are per call — the harness divides out its inner repetition count — with
20–30 repeats per cell. All quantities reported below are medians over those
repeats.

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import altair as alt
import polars as pl
from IPython.display import HTML, display

# Flip to "dark" for the dark palette; every colour below re-derives from it.
THEME = "light"

DATA_ROOT = Path("data")
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'

PALETTE = {
    "light": {
        "surface": "#fcfcfb", "plane": "#f9f9f7",
        "ink": "#0b0b0b", "ink2": "#52514e", "muted": "#898781",
        "grid": "#e1e0d9", "axis": "#c3c2b7",
        "series": ["#2a78d6", "#eb6834", "#1baf7a"],
        "shade": "#9ec5f4",     # second shade of series 1, for dumbbells
        "context": "#c3c2b7",   # de-emphasis grey for "everything else"
        "ramp": ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5",
                 "#256abf", "#184f95", "#0d366b"],
    },
    "dark": {
        "surface": "#1a1a19", "plane": "#0d0d0d",
        "ink": "#ffffff", "ink2": "#c3c2b7", "muted": "#898781",
        "grid": "#2c2c2a", "axis": "#383835",
        "series": ["#3987e5", "#d95926", "#199e70"],
        "shade": "#184f95",
        "context": "#52514e",
        "ramp": ["#0d366b", "#184f95", "#256abf", "#3987e5",
                 "#6da7ec", "#9ec5f4", "#cde2fb"],
    },
}
P = PALETTE[THEME]


@alt.theme.register("quadbench", enable=True)
def _quadbench_theme() -> dict:
    """Thin marks, hairline solid chrome, recessive axes, validated hues."""
    return {
        "config": {
            "background": P["surface"],
            "font": FONT,
            "view": {"stroke": None, "continuousWidth": 620, "continuousHeight": 300},
            "axis": {
                "labelFont": FONT, "titleFont": FONT,
                "labelColor": P["muted"], "titleColor": P["ink2"],
                "labelFontSize": 11, "titleFontSize": 11, "titleFontWeight": 500,
                "titlePadding": 8,
                "domainColor": P["axis"], "domainWidth": 1,
                "tickColor": P["axis"], "tickSize": 4,
                "gridColor": P["grid"], "gridWidth": 1, "gridDash": [],
            },
            "legend": {
                "labelFont": FONT, "titleFont": FONT,
                "labelColor": P["ink2"], "titleColor": P["ink2"],
                "labelFontSize": 11, "titleFontSize": 11, "titleFontWeight": 500,
                "symbolType": "circle", "symbolSize": 90, "orient": "top",
                "direction": "horizontal", "offset": 8, "titlePadding": 10,
            },
            "title": {
                "font": FONT, "color": P["ink"], "fontSize": 15, "fontWeight": 600,
                "subtitleFont": FONT, "subtitleColor": P["ink2"],
                "subtitleFontSize": 11.5, "subtitlePadding": 8,
                "anchor": "start", "offset": 14, "dy": -4,
            },
            "header": {
                "labelFont": FONT, "titleFont": FONT,
                "labelColor": P["ink2"], "titleColor": P["ink2"],
                "labelFontSize": 11.5, "labelFontWeight": 600, "titleFontSize": 11,
            },
            "range": {
                "category": P["series"],
                "heatmap": P["ramp"],
                "ramp": P["ramp"],
            },
            "bar": {"cornerRadiusEnd": 4, "discreteBandSize": 15},
            "point": {"size": 95, "filled": True,
                      "stroke": P["surface"], "strokeWidth": 2, "opacity": 1},
            "line": {"strokeWidth": 2, "strokeCap": "round", "strokeJoin": "round"},
            "rule": {"strokeWidth": 1},
            "text": {"font": FONT, "fontSize": 11, "color": P["ink2"]},
        }
    }


def table_view(df: pl.DataFrame, label: str = "Table view") -> HTML:
    """The WCAG-clean twin of a chart: every plotted value, as text."""
    return HTML(
        f"<details style='font:12px {FONT};color:{P['ink2']};margin:2px 0 18px'>"
        f"<summary style='cursor:pointer;padding:4px 0'>{label} "
        f"({df.height:,} rows)</summary>{df._repr_html_()}</details>"
    )


def figure(chart, table: pl.DataFrame | None = None, label: str = "Table view"):
    display(chart)
    if table is not None:
        display(table_view(table, label))


# Altair's default renderer embeds the spec plus a jsdelivr script tag, so saved
# outputs need a network connection to draw. alt.renderers.enable("mimetype")
# emits the raw Vega-Lite JSON instead — smaller, offline, but it relies on the
# viewer (JupyterLab, VS Code) shipping its own Vega-Lite 6 renderer.
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_width_chars(160)
alt.data_transformers.enable("default", max_rows=20000)

DataTransformerRegistry.enable('default')

## Loading

`time_us` is the only directly measured quantity. Two derived quantities carry
most of the analysis:

- **ns/element** — `time_us * 1000 / n_elem`, the per-element cost, which is
  comparable across problem sizes.
- **GB/s** — `(reads * operand_bytes + result_bytes) * n_elem / seconds`, where
  `reads` is 1 for the unary operations (`sqrt`, `exp`, `cos`) and 2 otherwise,
  and `result_bytes` follows from the recorded result dtype rather than the
  operand dtype. The distinction is material: `cos` applied to `int32` operands
  is computed in `float64`, so the corresponding traffic is 4 bytes in and
  8 bytes out.

In [2]:
# Storage width in bytes. Hardcoded rather than asked of the local NumPy: the runs
# were collected on x86 Linux, where longdouble is the 80-bit x87 type in a 16-byte
# slot — a machine reading this notebook may disagree.
ITEMSIZE = {
    "int8": 1, "int16": 2, "int32": 4, "int64": 8,
    "uint8": 1, "uint16": 2, "uint32": 4, "uint64": 8,
    "float16": 2, "float32": 4, "float64": 8, "float128": 16,
    "longdouble64": 16, "quad-sleef": 16,
    "QuadPrecDType(backend='sleef')": 16,
    "GPU fp16": 2, "GPU fp32": 4, "GPU fp64": 8,
}
UNARY = ["sqrt", "exp", "cos"]

DTYPE_ORDER = ["int8", "int16", "int32", "int64",
               "uint8", "uint16", "uint32", "uint64",
               "float16", "float32", "float64", "longdouble64", "quad-sleef"]
OP_ORDER = ["add", "mul", "div", "muladd", "muladd_accum", "muladd_fused",
            "sqrt", "exp", "cos",
            "matmul", "matmul_explicit", "matmul_explicit_fused"]


def load_run(run: str) -> tuple[pl.DataFrame, dict]:
    """Per-repeat samples, enriched with element counts, byte widths and rates."""
    run_dir = DATA_ROOT / run
    df = pl.read_parquet(run_dir / f"{run}.parquet")
    meta = json.loads((run_dir / f"{run}.json").read_text())

    if "n" not in df.columns:  # single-size runs drop the n field from their keys
        df = df.with_columns(pl.lit(meta["sizes"][0], dtype=pl.Int64).alias("n"))

    n_elem = {int(size): block["n_elem"] for size, block in meta["per_size"].items()}
    promotions = pl.DataFrame(
        [
            {"n": int(size), "operation": key.split("__")[0],
             "implementation": key.split("__")[1], "result_dtype": res}
            for size, block in meta["per_size"].items()
            for key, res in block.get("result_dtypes", {}).items()
        ],
        schema={"n": pl.Int64, "operation": pl.String,
                "implementation": pl.String, "result_dtype": pl.String},
    )

    return (
        df.join(promotions, on=["n", "operation", "implementation"], how="left")
        .with_columns(
            pl.lit(run).alias("run"),
            pl.col("n").replace_strict(n_elem).alias("n_elem"),
            # the GPU harness records no promotions; nothing it runs promotes
            pl.col("result_dtype").fill_null(pl.col("implementation")),
        )
        .with_columns(
            pl.col("implementation").replace_strict(ITEMSIZE).alias("operand_bytes"),
            pl.col("result_dtype").replace_strict(ITEMSIZE).alias("result_bytes"),
            pl.when(pl.col("operation").is_in(UNARY)).then(1).otherwise(2).alias("reads"),
            (pl.col("result_dtype") != pl.col("implementation")).alias("promoted"),
        )
        .with_columns(
            (pl.col("time_us") * 1e3 / pl.col("n_elem")).alias("ns_elem"),
            (
                (pl.col("reads") * pl.col("operand_bytes") + pl.col("result_bytes"))
                * pl.col("n_elem") / (pl.col("time_us") * 1e-6) / 1e9
            ).alias("gbs"),
        ),
        meta,
    )


KEYS = ["run", "n", "n_elem", "operation", "implementation",
        "destination", "cache_state"]


def cells(df: pl.DataFrame) -> pl.DataFrame:
    """One row per measured cell: median plus the interquartile spread."""
    return (
        df.group_by(KEYS)
        .agg(
            pl.col("time_us").median().alias("us"),
            pl.col("time_us").quantile(0.25).alias("us_lo"),
            pl.col("time_us").quantile(0.75).alias("us_hi"),
            pl.col("ns_elem").median().alias("ns_elem"),
            pl.col("gbs").median().alias("gbs"),
            pl.col("result_dtype").first(),
            pl.col("promoted").first(),
            pl.col("time_us").is_not_null().sum().alias("samples"),
        )
        .sort(KEYS)
    )


sweep_raw, sweep_meta = load_run("cpu_sweep")
dram_raw, dram_meta = load_run("cpu_dram")
gpu_raw, gpu_meta = load_run("gpu_sweep")

sweep, dram, gpu = cells(sweep_raw), cells(dram_raw), cells(gpu_raw)
everything = pl.concat([sweep, dram, gpu])
everything.head(5)

run,n,n_elem,operation,implementation,destination,cache_state,us,us_lo,us_hi,ns_elem,gbs,result_dtype,promoted,samples
str,i64,i64,str,str,str,str,f64,f64,f64,f64,f64,str,bool,u32
"""cpu_sweep""",500,2000,"""add""","""float16""","""alloc""","""cold""",12.653133,12.646402,12.669868,6.326566,0.948382,"""float16""",false,30
"""cpu_sweep""",500,2000,"""add""","""float16""","""alloc""","""hot""",12.591369,12.580333,12.604403,6.295684,0.953034,"""float16""",false,30
"""cpu_sweep""",500,2000,"""add""","""float16""","""out""","""cold""",12.596365,12.582998,12.617065,6.298182,0.952656,"""float16""",false,30
"""cpu_sweep""",500,2000,"""add""","""float16""","""out""","""hot""",12.520235,12.508865,12.528869,6.260118,0.958448,"""float16""",false,30
"""cpu_sweep""",500,2000,"""add""","""float32""","""alloc""","""cold""",0.883287,0.879787,0.88686,0.441643,27.171247,"""float32""",false,30


In [3]:
def stat_tile(label: str, value: str, note: str) -> str:
    return (
        f"<div style='flex:1 1 190px;min-width:170px;background:{P['surface']};"
        f"border:1px solid {P['grid']};border-radius:10px;padding:14px 16px'>"
        f"<div style='font:500 11.5px {FONT};color:{P['muted']};"
        f"letter-spacing:.02em'>{label}</div>"
        f"<div style='font:600 30px {FONT};color:{P['ink']};margin:6px 0 2px'>{value}</div>"
        f"<div style='font:400 11.5px {FONT};color:{P['ink2']}'>{note}</div></div>"
    )


measured = everything.filter(pl.col("us").is_not_null()).height
planned = everything.height
gpu_dev = gpu_meta["device"]

display(HTML(
    f"<div style='display:flex;gap:12px;flex-wrap:wrap;font:{FONT};"
    f"background:{P['plane']};padding:14px;border-radius:12px'>"
    + stat_tile("Cells measured", f"{measured:,}",
                f"of {planned:,} planned · {planned - measured} skipped by the harness")
    + stat_tile("Timing samples", f"{len(sweep_raw) + len(dram_raw) + len(gpu_raw):,}",
                "20–30 repeats per cell, medians throughout")
    + stat_tile("CPU", "x86-64 · 1 thread",
                f"AVX2 usable ({', '.join(sweep_meta['env']['cpu_dispatch_usable'])}),"
                f" OpenBLAS")
    + stat_tile("GPU", gpu_dev["name"].replace("NVIDIA ", ""),
                f"{gpu_dev['sms']} SMs · {gpu_dev['peak_hbm_gbs']:,.0f} GB/s peak HBM")
    + "</div>"
))

---

## Part 1 — `cpu_sweep`: precision and the cache hierarchy

The design is 13 dtypes × 10 operations × {`alloc`, `out`} × {`hot`, `cold`} ×
3 sizes. This is the only run containing software floating point, and therefore
the run in which the cost of extended precision becomes visible.

The first question is what the overall landscape looks like. The figure below
shows the `hot`, `out=` quadrant at the largest size (n = 64 000, i.e. 256 000
elements): operands are cache-resident and no allocation intervenes, so the
remaining variation is attributable to the arithmetic itself.

In [4]:
HOT_OUT = (pl.col("destination") == "out") & (pl.col("cache_state") == "hot")
BIG = pl.col("n") == 64_000

landscape = (
    sweep.filter(HOT_OUT & BIG)
    .select("operation", "implementation", "ns_elem", "result_dtype", "promoted")
    .sort("operation", "implementation")
)

fig1 = (
    alt.Chart(landscape)
    .mark_rect(stroke=P["surface"], strokeWidth=2)  # the 2px surface gap, not a border
    .encode(
        x=alt.X("implementation:N", sort=DTYPE_ORDER, title=None,
                axis=alt.Axis(labelAngle=-40, labelLimit=110)),
        y=alt.Y("operation:N", sort=OP_ORDER, title=None),
        color=alt.Color(
            "ns_elem:Q",
            scale=alt.Scale(type="log", range=P["ramp"]),
            legend=alt.Legend(title="ns / element", format="~g",
                              values=[0.1, 0.3, 1, 3, 10, 30, 100],
                              gradientLength=240, orient="right", direction="vertical"),
        ),
        tooltip=[
            alt.Tooltip("operation:N", title="op"),
            alt.Tooltip("implementation:N", title="dtype"),
            alt.Tooltip("result_dtype:N", title="computes in"),
            alt.Tooltip("ns_elem:Q", title="ns/element", format=".3f"),
        ],
    )
    .properties(
        width=520, height=310,
        title=alt.Title(
            "The cost of one element, by dtype and operation",
            subtitle="cpu_sweep · n = 64 000 · cache-resident, preallocated output · "
                     "log colour scale · blank = never run",
        ),
    )
)
figure(fig1, landscape.pivot("implementation", index="operation", values="ns_elem"),
       "Table view — ns per element")

alt.Chart(...)

operation,float16,float32,float64,int16,int32,int64,int8,longdouble64,quad-sleef,uint16,uint32,uint64,uint8
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""add""",5.998391,0.248638,0.650674,0.127026,0.298339,0.682356,0.05625,3.751268,28.796561,0.121054,0.283498,0.671407,0.057812
"""cos""",8.89775,1.629873,13.696295,1.677426,11.455238,11.701338,10.239508,188.230137,136.858086,1.71034,11.514494,11.782396,10.519641
"""div""",6.005355,0.285253,0.64406,0.928088,0.928457,1.490502,0.835916,3.789484,78.028256,0.867674,1.154305,1.600147,0.881004
"""exp""",6.287893,1.458293,4.61618,1.365982,4.825082,5.071357,9.175961,27.648184,123.711701,1.438996,4.892809,5.123707,9.451205
"""matmul""",11.672535,18.249818,15.920854,2.964234,2.981805,2.871748,2.984371,11.134717,null,3.143623,2.841303,3.027639,2.940535
"""matmul_explicit""",19.241068,2.828955,4.362369,2.606783,2.912082,4.359609,2.576685,13.287652,116.612785,2.615475,2.933647,4.370961,2.566414
"""mul""",6.017467,0.283642,0.64677,0.123702,0.276673,0.64262,0.058278,3.763084,42.443598,0.124323,0.257823,0.649949,0.058238
"""muladd""",12.012039,0.582348,1.462461,0.265847,0.572934,1.416178,0.119119,7.490848,71.36399,0.256953,0.558856,1.447195,0.116485
"""muladd_accum""",12.016324,0.426001,1.176613,0.207095,0.429451,1.136301,0.090671,7.485113,71.305654,0.205848,0.470481,1.153857,0.088561


Two features of the grid organise the remainder of this section.

First, the software-quad column belongs to a different regime. `quad-sleef` is
not marginally slower than `float64`; it lies off the end of a logarithmic
scale. Software floating point in 128 bits costs one to two orders of magnitude,
and `longdouble` — the x87 80-bit type, held in a 16-byte slot — falls between
the two: hardware arithmetic, but scalar hardware that no SIMD unit will touch.

Second, the `div`, `exp` and `cos` rows are uniformly expensive, including in the
integer columns. This does not reflect slow integer arithmetic. NumPy promotes
the operands before computing, so `cos` on `int32` is a `float64` cosine; those
cells measure a floating-point operation carrying an integer's label. Because
the metadata records the promotion, the resulting cost can be quantified.

In [5]:
baseline = (
    sweep.filter(HOT_OUT & BIG & (pl.col("implementation") == "float64"))
    .select("operation", pl.col("ns_elem").alias("f64_ns"))
)

FAMILY = pl.when(pl.col("implementation") == "quad-sleef").then(pl.lit("quad (SLEEF, software)")) \
           .when(pl.col("implementation") == "longdouble64").then(pl.lit("longdouble (x87, 80-bit)")) \
           .otherwise(pl.lit("hardware dtypes"))

tax = (
    sweep.filter(HOT_OUT & BIG)
    .join(baseline, on="operation")
    .with_columns((pl.col("ns_elem") / pl.col("f64_ns")).alias("vs_f64"), FAMILY.alias("family"))
    .filter(pl.col("vs_f64").is_not_null())
    .select("operation", "implementation", "family", "vs_f64", "ns_elem")
)

SOFT = ["quad (SLEEF, software)", "longdouble (x87, 80-bit)"]
context, highlight = tax.filter(~pl.col("family").is_in(SOFT)), tax.filter(pl.col("family").is_in(SOFT))

# Label placement is data-driven, not eyeballed: put the quad value on the left of
# its dot whenever longdouble sits close enough to the right to collide with it.
neighbours = highlight.pivot("implementation", index="operation", values="vs_f64")
quad_labels = (
    highlight.filter(pl.col("implementation") == "quad-sleef")
    .join(neighbours, on="operation")
    .with_columns(
        ((pl.col("longdouble64") / pl.col("quad-sleef")).is_between(1.0, 1.9))
        .alias("crowded")
    )
)

X2 = alt.X("vs_f64:Q", scale=alt.Scale(type="log", nice=False),
           axis=alt.Axis(title="cost relative to float64 (log)",
                         values=[0.25, 0.5, 1, 2, 5, 10, 25, 50, 100, 250],
                         labelExpr="format(datum.value, '.3~g') + '\u00d7'"))
Y2 = alt.Y("operation:N", sort=OP_ORDER, title=None)

unity = alt.Chart(pl.DataFrame({"x": [1.0]})).mark_rule(
    color=P["ink2"], strokeWidth=1).encode(x=X2)
unity_t = alt.Chart(pl.DataFrame({"x": [1.0], "t": ["float64"]})).mark_text(
    align="center", fontSize=11, color=P["ink2"]).encode(
    x=X2, y=alt.value(9), text="t:N")
ctx_dots = alt.Chart(context).mark_point(
    size=70, filled=True, color=P["context"], opacity=0.9).encode(
    x=X2, y=Y2,
    tooltip=[alt.Tooltip("implementation:N", title="dtype"),
             alt.Tooltip("vs_f64:Q", title="x float64", format=".2f")])
soft_dots = alt.Chart(highlight).mark_point(size=150, filled=True).encode(
    x=X2, y=Y2,
    color=alt.Color("family:N", scale=alt.Scale(domain=SOFT, range=P["series"][:2]),
                    legend=alt.Legend(title=None)),
    tooltip=[alt.Tooltip("implementation:N", title="dtype"),
             alt.Tooltip("vs_f64:Q", title="x float64", format=".1f"),
             alt.Tooltip("ns_elem:Q", title="ns/element", format=".2f")])
lab_right = alt.Chart(quad_labels.filter(~pl.col("crowded"))).mark_text(
    align="left", dx=12, fontSize=11, color=P["ink2"]).encode(
    x=X2, y=Y2, text=alt.Text("vs_f64:Q", format=".0f"))
lab_left = alt.Chart(quad_labels.filter(pl.col("crowded"))).mark_text(
    align="right", dx=-12, fontSize=11, color=P["ink2"]).encode(
    x=X2, y=Y2, text=alt.Text("vs_f64:Q", format=".0f"))

fig2 = (unity + unity_t + ctx_dots + soft_dots + lab_right + lab_left).properties(
    width=560, height=300,
    title=alt.Title("Cost relative to float64, by operation",
                    subtitle="cpu_sweep · n = 64 000 · hot, out= · grey = the 11 hardware dtypes, "
                             "labelled values = quad's multiple of float64"),
)
figure(fig2, highlight.sort("operation", "implementation"), "Table view — soft-float slowdowns")

alt.LayerChart(...)

operation,implementation,family,vs_f64,ns_elem
str,str,str,f64,f64
"""add""","""longdouble64""","""longdouble (x87, 80-bit)""",5.765204,3.751268
"""add""","""quad-sleef""","""quad (SLEEF, software)""",44.256522,28.796561
"""cos""","""longdouble64""","""longdouble (x87, 80-bit)""",13.743143,188.230137
"""cos""","""quad-sleef""","""quad (SLEEF, software)""",9.992344,136.858086
"""div""","""longdouble64""","""longdouble (x87, 80-bit)""",5.883748,3.789484
"""div""","""quad-sleef""","""quad (SLEEF, software)""",121.150679,78.028256
"""exp""","""longdouble64""","""longdouble (x87, 80-bit)""",5.989408,27.648184
"""exp""","""quad-sleef""","""quad (SLEEF, software)""",26.799585,123.711701
"""matmul""","""longdouble64""","""longdouble (x87, 80-bit)""",0.699379,11.134717


The grey cloud is the informative part of the chart: the eleven hardware dtypes
all lie within a small factor of `float64`, which is the expected signature of
arithmetic performed by the FPU. The two highlighted series lie well outside that
range.

Quad's multiple of `float64` is smallest on `div` and on the transcendental
functions. This is not because quad is comparatively fast on those operations,
but because `float64` division and `cos` are themselves slow enough to narrow the
ratio; both terms of the ratio vary.

### Vectorisation of the float64 path

The harness supports a direct diagnostic. On cache-resident data, `float64`
should cost approximately 2× `float32` if both paths are vectorised, since AVX2
accommodates half as many `float64` lanes. A ratio substantially above 2
indicates that the `float64` loop has left the vector unit and fallen back to
scalar libm.

In [6]:
ratio = (
    sweep.filter(HOT_OUT & BIG & pl.col("implementation").is_in(["float32", "float64"]))
    .pivot("implementation", index="operation", values="us")
    .drop_nulls()
    .with_columns((pl.col("float64") / pl.col("float32")).alias("ratio"))
    .with_columns(pl.when(pl.col("ratio") > 2.6).then(pl.lit("scalar fallback"))
                    .otherwise(pl.lit("vectorised")).alias("verdict"))
    .sort("ratio", descending=True)
)

bars = alt.Chart(ratio).mark_bar(color=P["series"][0], size=14).encode(
    x=alt.X("ratio:Q", title="float64 time / float32 time",
            scale=alt.Scale(domainMin=0), axis=alt.Axis(tickCount=5)),
    y=alt.Y("operation:N", sort="-x", title=None),
    tooltip=[alt.Tooltip("operation:N", title="op"),
             alt.Tooltip("ratio:Q", title="f64/f32", format=".2f"),
             alt.Tooltip("float32:Q", title="float32 us", format=".2f"),
             alt.Tooltip("float64:Q", title="float64 us", format=".2f")])
lane_line = alt.Chart(pl.DataFrame({"x": [2.0]})).mark_rule(
    color=P["ink2"], strokeWidth=1).encode(x="x:Q")
lane_text = alt.Chart(pl.DataFrame({"x": [2.0], "t": ["2.0 — half the lanes, both paths vectorised"]})).mark_text(
    align="left", dx=6, fontSize=11, color=P["ink2"]).encode(
    x="x:Q", y=alt.value(248), text="t:N")
val = bars.mark_text(align="left", dx=6, fontSize=11, color=P["ink2"]).encode(
    text=alt.Text("ratio:Q", format=".2f"))

fig3 = (bars + val + lane_line + lane_text).properties(
    width=520, height=260,
    title=alt.Title("float64 / float32 time ratio, by operation",
                    subtitle="cpu_sweep · n = 64 000 · hot, out= · a ratio above 2 indicates the "
                             "float64 loop is not using the vector unit"),
)
figure(fig3, ratio.select("operation", "float32", "float64", "ratio", "verdict"),
       "Table view — f64/f32 ratios")

alt.LayerChart(...)

operation,float32,float64,ratio,verdict
str,f64,f64,f64,str
"""cos""",417.247502,3506.251465,8.403289,"""scalar fallback"""
"""sqrt""",95.783005,320.610503,3.347259,"""scalar fallback"""
"""exp""",373.322953,1181.742002,3.165468,"""scalar fallback"""
"""muladd_accum""",109.056331,301.213004,2.761995,"""scalar fallback"""
"""add""",63.651206,166.572499,2.616957,"""scalar fallback"""
"""muladd""",149.080995,374.390016,2.51132,"""vectorised"""
"""mul""",72.61234,165.573016,2.280232,"""vectorised"""
"""div""",73.024664,164.879253,2.257857,"""vectorised"""
"""matmul_explicit""",724.212499,1116.766478,1.542043,"""vectorised"""


### The cost of allocation as a function of size

Every cell was also run in an `alloc` variant, in which the result array is
allocated afresh rather than written into a preallocated buffer. Subtracting the
two variants isolates the cost of allocation. The expected form is a fixed cost
that becomes negligible as arrays grow; the measurements do not follow that
form.

In [7]:
FEATURED = ["float64", "int64", "quad-sleef"]

alloc_cost = (
    sweep.filter((pl.col("cache_state") == "cold") & (pl.col("operation") == "add"))
    .pivot("destination", index=["implementation", "n", "n_elem"], values="us")
    .drop_nulls()
    .with_columns(
        (pl.col("alloc") - pl.col("out")).alias("penalty_us"),
        ((pl.col("alloc") - pl.col("out")) / pl.col("out") * 100).alias("penalty_pct"),
    )
    .sort("implementation", "n")
)
rest = alloc_cost.filter(~pl.col("implementation").is_in(FEATURED))
shown = alloc_cost.filter(pl.col("implementation").is_in(FEATURED))

XN = alt.X("n_elem:Q", scale=alt.Scale(type="log"), title="elements (log)",
           axis=alt.Axis(values=[2000, 16000, 256000], format="~s"))
YP = alt.Y("penalty_pct:Q", title="alloc cost, % of the out= time")

zero = alt.Chart(pl.DataFrame({"y": [0.0]})).mark_rule(
    color=P["ink2"], strokeWidth=1).encode(y="y:Q")
ctx_lines = alt.Chart(rest).mark_line(color=P["context"], strokeWidth=1.5).encode(
    x=XN, y=YP, detail="implementation:N")
hi_lines = alt.Chart(shown).mark_line().encode(
    x=XN, y=YP,
    color=alt.Color("implementation:N", sort=FEATURED,
                    scale=alt.Scale(domain=FEATURED, range=P["series"]),
                    legend=alt.Legend(title=None)))
hi_dots = hi_lines.mark_point(size=95, filled=True).encode(
    tooltip=[alt.Tooltip("implementation:N", title="dtype"),
             alt.Tooltip("n_elem:Q", title="elements", format=","),
             alt.Tooltip("penalty_pct:Q", title="alloc cost %", format=".1f"),
             alt.Tooltip("penalty_us:Q", title="alloc cost us", format=".2f")])
hi_labels = alt.Chart(shown.filter(pl.col("n") == 64_000)).mark_text(
    align="left", dx=11, fontSize=11, color=P["ink2"]).encode(
    x=XN, y=YP, text="implementation:N")

fig4 = (zero + ctx_lines + hi_lines + hi_dots + hi_labels).properties(
    width=470, height=300,
    title=alt.Title("Cost of allocating the output, by problem size",
                    subtitle="cpu_sweep · add · cold · grey = the other ten dtypes · "
                             "below zero means the allocating form was the faster one"),
).configure_view(clip=False)
figure(fig4, alloc_cost.select("n", "n_elem", "implementation", "out", "alloc",
                               "penalty_us", "penalty_pct"),
       "Table view — allocation cost by size")

alt.LayerChart(...)

n,n_elem,implementation,out,alloc,penalty_us,penalty_pct
i64,i64,str,f64,f64,f64,f64
500,2000,"""float16""",12.596365,12.653133,0.056768,0.45067
4000,16000,"""float16""",97.113007,97.370998,0.257991,0.265661
64000,256000,"""float16""",1540.025492,1541.919512,1.894019,0.122986
500,2000,"""float32""",0.8266,0.883287,0.056687,6.85784
4000,16000,"""float32""",3.481018,3.632037,0.151019,4.33836
64000,256000,"""float32""",92.294998,86.637752,-5.657246,-6.129526
500,2000,"""float64""",1.159435,1.215317,0.055882,4.819735
4000,16000,"""float64""",7.326944,7.981618,0.654674,8.935151
64000,256000,"""float64""",228.151504,161.938515,-66.21299,-29.0215


At 2 000 elements allocation costs what one would expect: roughly 10% on top of
a cheap operation, and nothing measurable on an expensive one (`quad-sleef` is
flat at 0.1%, since 44× the arithmetic renders the allocator immaterial). At
256 000 elements, however, the 8-byte types have gone negative: `int64` and
`float64` are approximately 30% faster when they allocate a fresh output than
when they write into a caller-supplied buffer.

The effect is real and it inverts the usual guidance, so it warrants a careful
statement. The most plausible mechanism is read-for-ownership: writing into an
existing 2 MB buffer requires each cache line to be fetched before it is
overwritten, whereas a freshly mapped page can skip that fetch. This benchmark
was not designed to isolate the mechanism — no cache-miss counters were
collected — so the mechanism should be treated as a hypothesis and the change of
sign as the measurement. Part 2 repeats the comparison at 64 MB, where the sign
reverts.

### Locality: hot versus cold

The remaining factor is locality. `hot` reuses a single operand pair; `cold`
cycles through distinct pairs so that nothing remains resident.

In [8]:
locality = (
    sweep.filter((pl.col("destination") == "out")
                 & (pl.col("implementation") == "float64")
                 & pl.col("operation").is_in(["add", "div", "exp"]))
    .select("operation", "n_elem", "cache_state", "ns_elem", "us")
    .sort("operation", "n_elem")
)

loc_order = ["hot", "cold"]
lines = alt.Chart().mark_line().encode(
    x=alt.X("n_elem:Q", scale=alt.Scale(type="log"), title="elements (log)",
            axis=alt.Axis(format="~s", values=[2000, 16000, 256000])),
    y=alt.Y("ns_elem:Q", title="ns / element", scale=alt.Scale(domainMin=0)),
    color=alt.Color("cache_state:N", sort=loc_order,
                    scale=alt.Scale(domain=loc_order, range=P["series"][:2]),
                    legend=alt.Legend(title=None)))
pts = lines.mark_point(size=90, filled=True).encode(
    tooltip=[alt.Tooltip("operation:N", title="op"),
             alt.Tooltip("n_elem:Q", title="elements", format=","),
             alt.Tooltip("cache_state:N", title="locality"),
             alt.Tooltip("ns_elem:Q", title="ns/element", format=".3f")])

fig5 = alt.layer(lines, pts).properties(width=185, height=210).facet(
    column=alt.Column("operation:N", title=None, sort=["add", "div", "exp"]),
    data=locality,
).resolve_scale(y="independent").properties(
    title=alt.Title("Per-element cost by locality, for three operations",
                    subtitle="cpu_sweep · float64 · out= · note the independent y-scales"),
)
figure(fig5, locality, "Table view — hot vs cold")

alt.FacetChart(...)

operation,n_elem,cache_state,ns_elem,us
str,i64,str,f64,f64
"""add""",2000,"""cold""",0.579718,1.159435
"""add""",2000,"""hot""",0.480525,0.961051
"""add""",16000,"""cold""",0.457934,7.326944
"""add""",16000,"""hot""",0.292345,4.677513
"""add""",256000,"""cold""",0.891217,228.151504
"""add""",256000,"""hot""",0.650674,166.572499
"""div""",2000,"""cold""",0.641465,1.28293
"""div""",2000,"""hot""",0.605763,1.211526
"""div""",16000,"""cold""",0.482362,7.717786


Two effects are superimposed in these panels. The U-shape reflects per-call
overhead: at 2 000 elements the fixed cost of entering the loop is spread over
too few elements, so the per-element cost is high; by 16 000 elements it has
amortised and the cost reaches its minimum; by 256 000 elements the operands no
longer fit in cache and the cost rises again.

The separation between the two lines is the effect of interest here. `add` is
cheap enough that operand fetch dominates, so `cold` costs measurably more once
the array outgrows cache: 0.89 against 0.65 ns/element at 256 000 elements.
`exp` is expensive enough that the fetch disappears into the arithmetic, and the
two lines coincide at every size. The pair of panels therefore separates
memory-bound from compute-bound behaviour, which is the subject of the next run.

---

## Part 2 — `cpu_dram`: operands resident in main memory

This run uses one size (n = 2 000 000, i.e. 8 million elements, a 64 GB
footprint across the operand set), four dtypes, and is pinned to a single NUMA
node with `numactl`. Nothing fits in any cache level, so the `cold` measurements
are effectively a bandwidth measurement expressed in arithmetic terms.

In [9]:
peak_line = (
    dram.filter((pl.col("destination") == "out") & (pl.col("cache_state") == "cold"))
    .select("gbs").max().item()
)
band = (
    dram.filter((pl.col("destination") == "out") & (pl.col("cache_state") == "cold"))
    .drop_nulls("gbs")
    .with_columns(
        pl.format("{} · {}", pl.col("operation"), pl.col("implementation")).alias("cell"),
        (pl.col("gbs") / peak_line * 100).alias("pct_of_plateau"),
    )
    .sort("gbs", descending=True)
    .select("cell", "operation", "implementation", "result_dtype", "gbs",
            "pct_of_plateau", "ns_elem", "us")
)

bw_bars = alt.Chart(band).mark_bar(color=P["series"][0], size=9).encode(
    y=alt.Y("cell:N", sort="-x", title=None, axis=alt.Axis(labelLimit=170)),
    x=alt.X("gbs:Q", title="effective bandwidth, GB/s"),
    tooltip=[alt.Tooltip("cell:N", title="cell"),
             alt.Tooltip("result_dtype:N", title="computes in"),
             alt.Tooltip("gbs:Q", title="GB/s", format=".1f"),
             alt.Tooltip("us:Q", title="us/call", format=".0f")])
plateau = alt.Chart(pl.DataFrame({"x": [peak_line]})).mark_rule(
    color=P["ink2"], strokeWidth=1).encode(x="x:Q")
plateau_t = alt.Chart(pl.DataFrame({"x": [peak_line], "t": [f"{peak_line:.1f} GB/s — highest observed"]})).mark_text(
    align="right", dx=-6, fontSize=11, color=P["ink2"]).encode(
    x="x:Q", y=alt.value(508), text="t:N")

fig6 = (bw_bars + plateau + plateau_t).properties(
    width=470, height=520,
    title=alt.Title("Effective bandwidth across 34 measured cells",
                    subtitle="cpu_dram · 8M elements · cold, out= · bars that fall short are "
                             "compute-bound, not memory-bound"),
)
figure(fig6, band, "Table view — effective bandwidth")

alt.LayerChart(...)

cell,operation,implementation,result_dtype,gbs,pct_of_plateau,ns_elem,us
str,str,str,str,f64,f64,f64,f64
"""add · int64""","""add""","""int64""","""int64""",27.678687,100.0,0.867095,6936.756487
"""add · float32""","""add""","""float32""","""float32""",27.535332,99.482076,0.43581,3486.483009
"""mul · int64""","""mul""","""int64""","""int64""",27.440105,99.138032,0.874634,6997.068005
"""mul · float64""","""mul""","""float64""","""float64""",27.403801,99.006871,0.875797,7006.372005
"""mul · float32""","""mul""","""float32""","""float32""",27.247844,98.443413,0.440426,3523.410007
"""div · float64""","""div""","""float64""","""float64""",27.18569,98.218858,0.882838,7062.700024
"""add · int32""","""add""","""int32""","""int32""",27.057825,97.756897,0.443495,3547.956992
"""add · float64""","""add""","""float64""","""float64""",27.048414,97.722895,0.887301,7098.405011
"""mul · int32""","""mul""","""int32""","""int32""",26.404353,95.395975,0.454587,3636.694513


The upper portion of the chart is flat, and that flatness is the finding: `add`,
`mul`, `muladd` and related operations, across all four dtypes, fall within a few
percent of a single value. They are not computing at different rates; they are
all limited by the same DRAM channels, and the arithmetic is negligible by
comparison.

The bars that fall away below that level are the operations that do not fit this
model — `exp`, `div` and the matrix multiplications — where the processor is
genuinely compute-limited. The chart is best read as a partition rather than as a
ranking.

### Allocation at 8 million elements

At this scale the change of sign observed in Part 1 is absent: allocation is a
cost in every cell. Its magnitude is predicted not by the operand dtype but by
the result width, since a fresh 64 MB buffer requires twice as many pages to be
faulted in as a 32 MB one. Because `div`, `sqrt`, `exp` and `cos` promote their
integer inputs to `float64`, an `int32` operand may fall into either group; the
chart below is therefore organised by result dtype rather than by input dtype.

In [10]:
dram_alloc = (
    dram.filter(pl.col("cache_state") == "cold")
    .pivot("destination", index=["operation", "implementation", "result_dtype"], values="us")
    .drop_nulls()
    .with_columns(
        (pl.col("alloc") - pl.col("out")).alias("penalty_us"),
        ((pl.col("alloc") - pl.col("out")) / pl.col("out") * 100).alias("penalty_pct"),
        pl.col("result_dtype").replace_strict(ITEMSIZE).alias("result_bytes"),
    )
    .sort("penalty_pct", descending=True)
)

dram_alloc = dram_alloc.with_columns(
    pl.format("{}-byte result", pl.col("result_bytes")).alias("width"),
    pl.when(pl.col("result_dtype") != pl.col("implementation"))
      .then(pl.lit("promoted before computing"))
      .otherwise(pl.lit("computed in its own dtype")).alias("promotion"),
)
dram_alloc = dram_alloc.with_columns(
    (0.5 + ((pl.int_range(pl.len()).over("width") % 7) - 3) * 0.11).alias("jitter")
)
medians = dram_alloc.group_by("width").agg(pl.col("penalty_pct").median().alias("med"))
prom_order = ["computed in its own dtype", "promoted before computing"]

YW = alt.Y("width:N", sort=["4-byte result", "8-byte result"], title=None)
XP = alt.X("penalty_pct:Q", title="allocation penalty, % of the out= time")

strip = alt.Chart(dram_alloc).mark_point(size=110, filled=True, opacity=0.95).encode(
    x=XP, y=YW,
    yOffset=alt.YOffset("jitter:Q", scale=alt.Scale(domain=[0, 1])),
    color=alt.Color("promotion:N", sort=prom_order,
                    scale=alt.Scale(domain=prom_order, range=P["series"][:2]),
                    legend=alt.Legend(title=None)),
    tooltip=[alt.Tooltip("operation:N", title="op"),
             alt.Tooltip("implementation:N", title="operand dtype"),
             alt.Tooltip("result_dtype:N", title="computes in"),
             alt.Tooltip("penalty_pct:Q", title="penalty %", format=".1f"),
             alt.Tooltip("penalty_us:Q", title="penalty us", format=".0f")])
med_tick = alt.Chart(medians).mark_tick(
    color=P["ink2"], thickness=2, size=74, opacity=1).encode(x="med:Q", y=YW)
med_text = alt.Chart(medians).mark_text(
    align="center", dy=-46, fontSize=11, color=P["ink2"]).encode(
    x="med:Q", y=YW, text=alt.Text("med:Q", format=".0f"))

fig7 = (strip + med_tick + med_text).properties(
    width=470, height=210,
    title=alt.Title("Allocation penalty grouped by result width",
                    subtitle="cpu_dram · cold · one dot per (op, dtype) · dark tick = group median · "
                             "the groups overlap, so this is a tendency, not a rule"),
)
figure(fig7, dram_alloc.select("operation", "implementation", "result_dtype",
                               "result_bytes", "out", "alloc", "penalty_us", "penalty_pct")
       .sort("result_bytes", "penalty_pct"),
       "Table view — allocation penalty at DRAM scale")

alt.LayerChart(...)

operation,implementation,result_dtype,result_bytes,out,alloc,penalty_us,penalty_pct
str,str,str,i64,f64,f64,f64,f64
"""mul""","""int32""","""int32""",4,3636.694513,3500.66548,-136.029033,-3.740458
"""exp""","""float32""","""float32""",4,9653.11451,9876.380995,223.266485,2.312896
"""add""","""int32""","""int32""",4,3547.956992,3631.092492,83.135499,2.343194
"""sqrt""","""float32""","""float32""",4,2942.235034,3046.411497,104.176463,3.540725
"""matmul_explicit""","""float32""","""float32""",4,27798.249997,29153.037525,1354.787528,4.873643
"""add""","""float32""","""float32""",4,3486.483009,3813.196032,326.713023,9.370848
"""matmul_explicit""","""int32""","""int32""",4,28288.940521,31493.121496,3204.180975,11.326621
"""matmul""","""int32""","""int32""",4,23814.638989,26682.932017,2868.293028,12.044243
"""cos""","""float32""","""float32""",4,12000.096525,14701.410488,2701.313962,22.510769


The group medians are 12% and 46% respectively — a four-fold difference that the
operand dtype alone would not have predicted. The groups do overlap
(`muladd_accum` on `float32` incurs 68%, exceeding most 8-byte cells), so the
relationship is a tendency rather than a rule, which is why the medians are drawn
explicitly rather than left to the eye.

### Cells excluded by the harness

`cpu_dram` was run with `--max-call-ms 50`, so the calibration pass excluded any
cell whose single call would exceed 50 ms. This removed 24 of the 160 planned
combinations, and those 24 are not a random subset. They are charted below
because a missing cell is easily mistaken for a zero.

In [11]:
coverage = (
    dram.filter((pl.col("destination") == "alloc") & (pl.col("cache_state") == "cold"))
    .with_columns(pl.when(pl.col("us").is_null()).then(pl.lit("skipped: over 50 ms/call"))
                    .otherwise(pl.lit("measured")).alias("state"))
    .select("operation", "implementation", "state", "us")
)

fig8 = (
    alt.Chart(coverage)
    .mark_rect(stroke=P["surface"], strokeWidth=2)
    .encode(
        x=alt.X("implementation:N", sort=["int32", "float32", "int64", "float64"],
                title=None, axis=alt.Axis(labelAngle=0)),
        y=alt.Y("operation:N", sort=OP_ORDER, title=None),
        color=alt.Color("state:N", sort=["measured", "skipped: over 50 ms/call"],
                        scale=alt.Scale(domain=["measured", "skipped: over 50 ms/call"],
                                        range=[P["series"][0], P["series"][1]]),
                        legend=alt.Legend(title=None)),
        tooltip=[alt.Tooltip("operation:N", title="op"),
                 alt.Tooltip("implementation:N", title="dtype"),
                 alt.Tooltip("state:N", title="status"),
                 alt.Tooltip("us:Q", title="alloc us/call", format=".0f")],
    )
    .properties(
        width=190, height=250,
        title=alt.Title("Coverage: cells excluded by the 50 ms per-call limit",
                        subtitle="cpu_dram · the gaps are the slowest cells, so any average "
                                 "across a row is biased toward faster values"),
    )
)
figure(fig8, coverage.filter(pl.col("state") != "measured").select("operation", "implementation"),
       "Table view — skipped cells")

alt.Chart(...)

operation,implementation
str,str
"""cos""","""float64"""
"""cos""","""int32"""
"""cos""","""int64"""
"""matmul""","""float32"""
"""matmul""","""float64"""
"""matmul_explicit""","""float64"""
"""matmul_explicit""","""int64"""


`cos` survives only in `float32`; for every other dtype it is promoted to
`float64` and exceeds the budget. An average of "cos across dtypes" in this run
would therefore report only the single inexpensive case. Gaps of this kind are
better marked on the figure than relegated to a footnote.

---

## Part 3 — `gpu_sweep`: an A100 at four sizes and three precisions

This run uses a different harness (`bench_gpu.py`, timed with CUDA events) and a
different machine: 108 SMs and 1 555 GB/s of HBM bandwidth. The factor of
interest is problem size. At 80 000 elements a GPU is dominated by launch
overhead, and it approaches its throughput limit only when there is enough work
to occupy it.

In [12]:
PEAK = gpu_meta["peak_hbm_gbs"]
sat = (
    gpu.filter((pl.col("destination") == "out") & (pl.col("cache_state") == "cold")
               & (pl.col("operation") == "add"))
    .with_columns((pl.col("gbs") / PEAK * 100).alias("pct_peak"))
    .select("implementation", "n_elem", "gbs", "pct_peak", "us")
    .sort("implementation", "n_elem")
)

gpu_order = ["GPU fp16", "GPU fp32", "GPU fp64"]
XG = alt.X("n_elem:Q", scale=alt.Scale(type="log"), title="elements (log)",
           axis=alt.Axis(values=[80_000, 1_000_000, 10_000_000, 100_000_000],
                         format="~s"))
sat_line = alt.Chart(sat).mark_line().encode(
    x=XG,
    y=alt.Y("pct_peak:Q", title="% of the 1 555 GB/s HBM peak",
            scale=alt.Scale(domain=[0, 100])),
    color=alt.Color("implementation:N", sort=gpu_order,
                    scale=alt.Scale(domain=gpu_order, range=P["series"]),
                    legend=alt.Legend(title=None)))
sat_pts = sat_line.mark_point(size=95, filled=True).encode(
    tooltip=[alt.Tooltip("implementation:N", title="dtype"),
             alt.Tooltip("n_elem:Q", title="elements", format=","),
             alt.Tooltip("gbs:Q", title="GB/s", format=".0f"),
             alt.Tooltip("pct_peak:Q", title="% of peak", format=".1f")])
# label at 1M, where the three series are furthest apart — at the right-hand end
# fp32 and fp64 converge and their labels would collide
sat_lab = alt.Chart(sat.filter(pl.col("n_elem") == 1_000_000)).mark_text(
    align="left", dx=8, dy=-11, fontSize=11, color=P["ink2"], fontWeight=500).encode(
    x=XG, y="pct_peak:Q", text=alt.Text("implementation:N"))

fig9 = (sat_line + sat_pts + sat_lab).properties(
    width=470, height=290,
    title=alt.Title("Fraction of peak HBM bandwidth against problem size",
                    subtitle="gpu_sweep · add · cold, out= · % of vendor peak HBM bandwidth · "
                             "labelled at the point of widest separation"),
).configure_view(clip=False)
figure(fig9, sat, "Table view — HBM saturation")

alt.LayerChart(...)

implementation,n_elem,gbs,pct_peak,us
str,i64,f64,f64,f64
"""GPU fp16""",80000,54.227992,3.486882,8.851692
"""GPU fp16""",1000000,616.980945,39.672129,9.7248
"""GPU fp16""",10000000,885.373639,56.929889,67.768
"""GPU fp16""",100000000,1007.279266,64.768471,595.664024
"""GPU fp32""",80000,109.306415,7.028447,8.782666
"""GPU fp32""",1000000,1175.958201,75.614596,10.204444
"""GPU fp32""",10000000,1151.012928,74.010605,104.255997
"""GPU fp32""",100000000,1341.945695,86.28766,894.223988
"""GPU fp64""",80000,216.576434,13.925954,8.865231


At 80 000 elements the card attains 4–14% of its own memory bandwidth: the
kernel completes before the machine is fully occupied. One order of magnitude
later, at one million elements, fp64 has already reached 84% of peak and the
curve has essentially finished climbing; the remaining two decades contribute
only a few further percentage points.

The ordering is the unexpected result: fp16 is the least efficient of the three
precisions, and remains so at every size. It is not slower in absolute terms — it
moves half the bytes of fp32 — but that is precisely the cause. Half the traffic
at a given element count means half the work available to saturate the bus, so
fp16 requires proportionally more elements to reach the same fraction of peak.
Narrow types therefore aggravate the launch-overhead problem rather than
alleviating it.

This leaves the question of whether fp64 is ever penalised arithmetically here.
A100 vector fp64 runs at half the rate of fp32 (9.7 against 19.5 TFLOP/s) and
fp16 is faster still, so a compute-bound operation should exhibit ratios set by
those rates, while a memory-bound operation should instead exhibit the ratio of
its byte traffic. Both predictions give 2.0 for fp64/fp32, which makes the
fp32/fp16 column the discriminating one.

In [13]:
gpu_ratio = (
    gpu.filter((pl.col("destination") == "out") & (pl.col("cache_state") == "hot")
               & (pl.col("n") == 25_000_000))
    .pivot("implementation", index="operation", values="us")
    .drop_nulls()
    .with_columns(
        (pl.col("GPU fp64") / pl.col("GPU fp32")).alias("fp64 / fp32"),
        (pl.col("GPU fp32") / pl.col("GPU fp16")).alias("fp32 / fp16"),
    )
    .sort("fp64 / fp32", descending=True)
)
gpu_long = gpu_ratio.unpivot(["fp64 / fp32", "fp32 / fp16"], index="operation",
                             variable_name="pair", value_name="ratio")

pair_order = ["fp64 / fp32", "fp32 / fp16"]
gbars = alt.Chart(gpu_long).mark_bar(size=9).encode(
    y=alt.Y("operation:N", sort=alt.EncodingSortField("ratio", op="max", order="descending"),
            title=None),
    yOffset=alt.YOffset("pair:N", sort=pair_order),
    x=alt.X("ratio:Q", title="time ratio"),
    color=alt.Color("pair:N", sort=pair_order,
                    scale=alt.Scale(domain=pair_order, range=P["series"][:2]),
                    legend=alt.Legend(title=None)),
    tooltip=[alt.Tooltip("operation:N", title="op"),
             alt.Tooltip("pair:N", title="ratio"),
             alt.Tooltip("ratio:Q", format=".2f")])
two = alt.Chart(pl.DataFrame({"x": [2.0]})).mark_rule(
    color=P["ink2"], strokeWidth=1).encode(
    x=alt.X("x:Q", scale=alt.Scale(domain=[0, 2.25], nice=False)))
two_t = alt.Chart(pl.DataFrame({"x": [2.0], "t": ["2.0 — half the bytes, half the time"]})).mark_text(
    align="right", dx=-5, fontSize=11, color=P["ink2"]).encode(
    x="x:Q", y=alt.value(278), text="t:N")

fig10 = (gbars + two + two_t).properties(
    width=430, height=290,
    title=alt.Title("Time ratios between precisions, by operation",
                    subtitle="gpu_sweep · 100M elements · hot, out= · each op's two ratios, "
                             "against the 2.0 expected of a purely bandwidth-bound op"),
)
figure(fig10, gpu_ratio, "Table view — precision ratios on the A100")

alt.LayerChart(...)

operation,GPU fp16,GPU fp32,GPU fp64,fp64 / fp32,fp32 / fp16
str,f64,f64,f64,f64,f64
"""matmul_explicit_fused""",450.016007,892.751992,1779.999971,1.993835,1.983823
"""muladd""",1176.83202,1776.864052,3541.328073,1.993021,1.509871
"""muladd_fused""",592.384011,893.712014,1776.495993,1.987772,1.50867
"""add""",593.344003,893.951982,1775.439978,1.986057,1.506634
"""mul""",594.000012,894.784003,1774.096012,1.982709,1.50637
"""div""",594.23998,894.400001,1767.232001,1.975886,1.505116
"""matmul_explicit""",3838.91201,6663.967848,13165.023804,1.975553,1.7359
"""exp""",592.25601,608.191997,1159.200013,1.905977,1.026907
"""sqrt""",592.17599,613.471985,1160.09599,1.891033,1.035962


The fp64/fp32 column is uninformative in the most useful way: the ratio is 2.00
for every elementwise operation, which is exactly what a bandwidth-bound kernel
moving twice the bytes should cost. On this card, at this size, fp64 incurs no
arithmetic penalty at all; its half-rate vector units never become the
bottleneck.

The fp32/fp16 column departs from that pattern. `add`, `mul`, `div` and the
muladd variants yield 1.5 rather than 2.0, and the unary operations (`sqrt`,
`exp`, `cos`) yield approximately 1.0, so halving the precision buys nothing.
Taken with the preceding figure, the interpretation is consistent: fp16 generates
too little traffic per element to keep the memory system occupied, so the bytes
it saves do not translate into time. `matmul` is the exception in both columns
(1.45 and 0.96), and it is the one operation dispatched to cuBLAS rather than to
an elementwise kernel.

### The three runs on a common axis

The three runs overlap in exactly one configuration — `add` on a 64-bit float,
streaming, into a preallocated buffer — which makes it the only defensible
cross-machine comparison. It covers seven problem sizes across four orders of
magnitude on a single metric.

In [14]:
CROSS = ((pl.col("destination") == "out") & (pl.col("cache_state") == "cold")
         & (pl.col("operation") == "add"))
crossover = (
    pl.concat([
        sweep.filter(CROSS & (pl.col("implementation") == "float64")),
        dram.filter(CROSS & (pl.col("implementation") == "float64")),
        gpu.filter(CROSS & (pl.col("implementation") == "GPU fp64")),
    ])
    .with_columns(
        pl.when(pl.col("run") == "gpu_sweep").then(pl.lit("A100 (GPU fp64)"))
          .otherwise(pl.lit("x86 CPU, 1 thread (float64)")).alias("machine")
    )
    .select("machine", "run", "n_elem", "ns_elem", "gbs", "us")
    .sort("machine", "n_elem")
)

mach_order = ["x86 CPU, 1 thread (float64)", "A100 (GPU fp64)"]
cl = alt.Chart(crossover).mark_line().encode(
    x=alt.X("n_elem:Q", scale=alt.Scale(type="log"), title="elements (log)",
            axis=alt.Axis(values=[2000, 16000, 256000, 8_000_000, 100_000_000],
                          format="~s")),
    y=alt.Y("ns_elem:Q", scale=alt.Scale(type="log"),
            title="ns per element (log)",
            axis=alt.Axis(values=[0.01, 0.03, 0.1, 0.3, 1], format="~g")),
    color=alt.Color("machine:N", sort=mach_order,
                    scale=alt.Scale(domain=mach_order, range=P["series"][:2]),
                    legend=alt.Legend(title=None)))
cp = cl.mark_point(size=95, filled=True).encode(
    tooltip=[alt.Tooltip("machine:N", title="machine"),
             alt.Tooltip("run:N", title="run"),
             alt.Tooltip("n_elem:Q", title="elements", format=","),
             alt.Tooltip("ns_elem:Q", title="ns/element", format=".4f"),
             alt.Tooltip("gbs:Q", title="GB/s", format=".1f")])

fig11 = (cl + cp).properties(
    width=520, height=300,
    title=alt.Title("The same operation on both machines, by problem size",
                    subtitle="add · float64 · cold, out= · the CPU line is two runs "
                             "(cpu_sweep, then cpu_dram at 8M elements)"),
)
figure(fig11, crossover, "Table view — CPU vs GPU")

alt.LayerChart(...)

machine,run,n_elem,ns_elem,gbs,us
str,str,i64,f64,f64,f64
"""A100 (GPU fp64)""","""gpu_sweep""",80000,0.110815,216.576434,8.865231
"""A100 (GPU fp64)""","""gpu_sweep""",1000000,0.018477,1298.894833,18.477333
"""A100 (GPU fp64)""","""gpu_sweep""",10000000,0.018987,1264.009541,189.871997
"""A100 (GPU fp64)""","""gpu_sweep""",100000000,0.017753,1351.91163,1775.263965
"""x86 CPU, 1 thread (float64)""","""cpu_sweep""",2000,0.579718,41.399472,1.159435
"""x86 CPU, 1 thread (float64)""","""cpu_sweep""",16000,0.457934,52.409343,7.326944
"""x86 CPU, 1 thread (float64)""","""cpu_sweep""",256000,0.891217,26.929476,228.151504
"""x86 CPU, 1 thread (float64)""","""cpu_dram""",8000000,0.887301,27.048414,7098.405011


Over the range in which both machines have data — 80 000 to 8 million elements —
the GPU is faster per element throughout, by roughly 4× at the small end and 45×
at the large end. No crossover appears in this data; if one exists, it lies below
80 000 elements, where the GPU run has no measurements and the CPU is still
paying its own per-call overhead. The defensible conclusion is that this
benchmark did not locate the size at which the CPU wins, not that no such size
exists.

---

## Conclusions

1. **Precision is not a uniformly spaced dial.** Every hardware dtype on this CPU
   falls within a small factor of `float64`; software quad is 27–121× more
   expensive, with x87 `longdouble` between them. There is no intermediate
   regime between arithmetic performed by the FPU and arithmetic performed by a
   library.

2. **Dtype labels do not identify what is executed.** `div`, `exp`, `cos` and
   `sqrt` promote their integer inputs to floating point before computing, so an
   "int32 cosine" is a `float64` cosine that has additionally paid for a
   conversion. Byte accounting in this notebook uses the recorded result dtype
   for exactly that reason, and in `cpu_dram` it is the result width, not the
   input width, that predicts the cost of allocation.

3. **Beyond cache, arithmetic ceases to be the operative variable.** In
   `cpu_dram` the inexpensive elementwise operations across four dtypes converge
   on a single bandwidth figure, 27.7 GB/s. The operations that fall short of it
   — `exp`, `div`, the matrix multiplications — are the ones for which arithmetic
   optimisation is appropriate; the remainder are memory problems in arithmetic
   dress.

4. **The cost of allocation changes sign with scale.** At 2 000 elements it is a
   cost of roughly 10%. At 256 000 elements it becomes a benefit of roughly 30%
   for 8-byte types, plausibly because writing to freshly mapped pages avoids
   read-for-ownership. At 8 million elements it is a cost again, 46% at the
   median for 8-byte results. These data do not support a general recommendation
   to preallocate; they support measuring at the intended size.

5. **The GPU exhibits the opposite limitation, and narrow types worsen it.** The
   A100 spends small problems almost entirely on launch overhead and reaches most
   of its bandwidth ceiling by approximately one million elements. fp64 costs
   exactly 2× fp32 throughout — pure byte traffic, with no arithmetic penalty —
   whereas fp16 never converts its halved traffic into halved time.

### Limitations

- `cpu_dram` is missing 24 of 160 cells and `cpu_sweep` 12 of 1 560, in both
  cases because the harness excluded the slowest cells. Aggregates over a dtype
  or an operation are therefore biased toward faster values unless they are
  restricted to complete cells.
- All reported quantities are medians of 20–30 repeats, single-threaded, on one
  NUMA node. The interquartile range is carried through `cells()` as
  `us_lo`/`us_hi` for checking the spread behind any individual claim.
- The CPU and GPU runs come from different harnesses using different timers
  (`perf_counter` and CUDA events respectively). The final figure compares
  end-to-end call cost, not kernel time.